# MPI — Sincronização e Operações Coletivas

**Disciplina:** DCE540 — Computação Paralela e Distribuída
**Curso:** Bacharelado em Ciência da Computação — UNIFAL-MG

## Objetivos da aula

- Compreender o conceito de **sincronização de processos** com `MPI_Barrier`.
- Conhecer e aplicar as principais **operações coletivas** do MPI:
  `MPI_Bcast`, `MPI_Scatter`, `MPI_Gather`, `MPI_Reduce`, `MPI_Allreduce` e `MPI_Allgather`.
- Praticar a implementação dessas operações em **Python (mpi4py)** e em **C**.

> Este notebook foi preparado para o Google Colab. Cada exemplo é escrito em um
> arquivo (`%%writefile`) e executado com `mpirun`, simulando múltiplos processos
> em um único ambiente.


## 0. Preparando o ambiente

Antes de tudo, instalamos o **Open MPI** (compilador `mpicc` e executor `mpirun`)
e a biblioteca **mpi4py**, que oferece os bindings de MPI para Python.

> **Nota sobre o `--oversubscribe`:** as máquinas virtuais do Google Colab
> costumam ter poucos núcleos de CPU disponíveis (muitas vezes apenas 2). Como
> os exemplos abaixo usam `-np 4` (4 processos), o Open MPI recusaria a
> execução por padrão, informando que não há "slots" suficientes. A flag
> `--oversubscribe`, presente em todos os comandos `mpirun` deste notebook,
> instrui o Open MPI a permitir mais processos do que núcleos físicos
> (eles serão intercalados pelo sistema operacional). Em um cluster real, com
> núcleos suficientes para o `-np` desejado, essa flag não é necessária.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq libopenmpi-dev openmpi-bin
!pip install -q mpi4py

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.1 MB/s eta 0:00:00


## 1. `MPI_Barrier` — Sincronização de processos

`MPI_Barrier` é uma **operação coletiva de sincronização**: diferente das
demais operações coletivas, ela **não transmite nenhum dado** — sua única
função é impor um ponto de encontro no código. Todo processo que a chama fica
bloqueado até que **todos os processos** do comunicador também tenham
chamado `MPI_Barrier`. Somente quando o último processo chega à barreira é
que a execução é liberada para todos, simultaneamente.

**Assinatura (C):**
```c
int MPI_Barrier(MPI_Comm comm);
```
- `comm`: o comunicador cujos processos devem ser sincronizados (em geral,
  `MPI_COMM_WORLD`, que reúne todos os processos da aplicação).

**Assinatura (mpi4py):**
```python
comm.Barrier()
```

**Quando usar:**
- Garantir que todos os processos terminaram uma fase antes de iniciar a
  próxima (ex.: medir com precisão o tempo de execução de um trecho
  paralelo, evitando que processos adiantados "vazem" para a etapa seguinte).
- Coordenar acessos a recursos externos compartilhados por todos os
  processos (arquivos, dispositivos, saída ordenada no terminal).
- Depurar programas paralelos, isolando em qual etapa cada processo está.

**Cuidado:** por ser uma operação **bloqueante e coletiva**, `MPI_Barrier`
tem custo de comunicação (proporcional ao número de processos) e pode gerar
espera ociosa (*idle time*) nos processos mais rápidos, que ficam parados
aguardando o mais lento. Deve ser usada apenas quando a sincronização é
realmente necessária para a corretude do programa.


In [ ]:
%%writefile barrier.py
from mpi4py import MPI
import time
import random
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

np.random.seed(42+rank)
# Cada processo "demora" um tempo diferente antes da barreira
time.sleep(0.05 * random.randint(10, 10 + size))
print(f"[Processo {rank}] cheguei na barreira", flush=True)

comm.Barrier()  # todos esperam aqui até o último processo chegar

print(f"[Processo {rank}] passei da barreira!", flush=True)

Overwriting barrier.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 40 python barrier.py

[Processo 1] cheguei na barreira
[Processo 28] cheguei na barreira
[Processo 3] cheguei na barreira
[Processo 8] cheguei na barreira
[Processo 6] cheguei na barreira
[Processo 16] cheguei na barreira
[Processo 24] cheguei na barreira
[Processo 37] cheguei na barreira
[Processo 19] cheguei na barreira
[Processo 7] cheguei na barreira
[Processo 34] cheguei na barreira
[Processo 2] cheguei na barreira
[Processo 17] cheguei na barreira
[Processo 29] cheguei na barreira
[Processo 15] cheguei na barreira
[Processo 27] cheguei na barreira
[Processo 12] cheguei na barreira
[Processo 21] cheguei na barreira
[Processo 38] cheguei na barreira
[Processo 39] cheguei na barreira
[Processo 20] cheguei na barreira
[Processo 5] cheguei na barreira
[Processo 31] cheguei na barreira
[Processo 13] cheguei na barreira
[Processo 30] cheguei na barreira
[Processo 25] cheguei na barreira
[Processo 32] cheguei na barreira
[Processo 11] cheguei na barreira
[Processo 35] cheguei na barreira
[Processo 33] cheguei

Versão equivalente em **C**:

In [ ]:
%%writefile barrier.c
#include <mpi.h>
#include <stdio.h>
#include <unistd.h>

int main(int argc, char** argv) {
    int rank, size;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    sleep(rank);  // cada processo "demora" um tempo diferente
    printf("[Processo %d] cheguei na barreira\n", rank);
    fflush(stdout);

    MPI_Barrier(MPI_COMM_WORLD);  // sincroniza todos os processos

    printf("[Processo %d] passei da barreira!\n", rank);

    MPI_Finalize();
    return 0;
}

Writing barrier.c


In [ ]:
!mpicc -o barrier barrier.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./barrier

[Processo 0] cheguei na barreira
[Processo 1] cheguei na barreira
[Processo 2] cheguei na barreira
[Processo 3] cheguei na barreira
[Processo 0] passei da barreira!
[Processo 1] passei da barreira!
[Processo 2] passei da barreira!
[Processo 3] passei da barreira!


## 2. Operações Coletivas

Operações coletivas envolvem **todos os processos de um comunicador** em uma
única chamada de função, ao contrário das operações ponto-a-ponto
(`Send`/`Recv`), que envolvem apenas dois processos por vez. Toda operação
coletiva deve ser chamada por **todos** os processos do comunicador — se
apenas alguns chamarem, o programa trava (*deadlock*).

Essas operações são implementadas internamente pela biblioteca MPI usando
algoritmos otimizados (tipicamente em árvore, com complexidade logarítmica no
número de processos), sendo muito mais eficientes do que obter o mesmo efeito
replicando várias mensagens ponto-a-ponto manualmente.

As principais operações coletivas de **movimentação de dados** e **redução**
são:

| Operação | O que faz |
|---|---|
| `MPI_Bcast` | Um processo (raiz) envia o **mesmo** dado para todos os demais |
| `MPI_Scatter` | Um processo (raiz) **distribui** partes diferentes de um vetor entre os processos |
| `MPI_Gather` | O processo raiz **coleta** dados enviados por todos os processos, na ordem do rank |
| `MPI_Reduce` | Combina valores de todos os processos (soma, máx, mín, ...) e entrega o resultado **à raiz** |
| `MPI_Allreduce` | Igual ao `Reduce`, mas o resultado é entregue **a todos** os processos |
| `MPI_Allgather` | Igual ao `Gather`, mas o resultado final vai **para todos** os processos |


### 2.1 `MPI_Bcast`

Distribui uma cópia do **mesmo** dado, presente no processo raiz, para todos
os demais processos do comunicador. Ao final da chamada, todos os processos —
inclusive a raiz — têm exatamente o mesmo conteúdo no buffer. É a forma mais
simples de levar uma informação de configuração, um parâmetro ou dado de
entrada lido por um único processo a todos os demais.

Nos exemplos abaixo, o dado é um **vetor** de inteiros, transmitido com a
interface baseada em buffers do NumPy (`Bcast`, com "B" maiúsculo) — a forma
eficiente de trocar vetores em mpi4py, pois evita serialização (`pickle`) e
opera diretamente sobre a memória contígua do array.

```c
int MPI_Bcast(void* buffer, int count, MPI_Datatype datatype,
              int root, MPI_Comm comm);
```
- `buffer`: endereço do dado — na raiz é o dado a enviar, nos demais é onde o
  dado recebido será armazenado (o mesmo buffer serve para enviar e receber).
- `count`: número de elementos no buffer.
- `datatype`: tipo dos elementos (ex.: `MPI_INT`, `MPI_DOUBLE`).
- `root`: rank do processo que possui o dado original.
- `comm`: comunicador envolvido na operação.

```python
comm.Bcast(vetor, root=0)
```


In [ ]:
%%writefile bcast.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

N = 5
if rank == 0:
    vetor = np.array([10, 20, 30, 40, 50], dtype='i')
else:
    vetor = np.empty(N, dtype='i')  # buffer vazio para receber

comm.Bcast(vetor, root=0)
print(f"[Processo {rank}] vetor recebido: {vetor}")

Writing bcast.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python bcast.py

[Processo 0] vetor recebido: [10 20 30 40 50]
[Processo 2] vetor recebido: [10 20 30 40 50]
[Processo 1] vetor recebido: [10 20 30 40 50]
[Processo 3] vetor recebido: [10 20 30 40 50]


Versão equivalente em **C**:

In [ ]:
%%writefile bcast.c
#include <mpi.h>
#include <stdio.h>

#define N 5

int main(int argc, char** argv) {
    int rank;
    int vetor[N];

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    if (rank == 0) {
        int original[N] = {10, 20, 30, 40, 50};
        for (int i = 0; i < N; i++) vetor[i] = original[i];
    }

    MPI_Bcast(vetor, N, MPI_INT, 0, MPI_COMM_WORLD);

    printf("[Processo %d] vetor recebido: ", rank);
    for (int i = 0; i < N; i++) printf("%d ", vetor[i]);
    printf("\n");

    MPI_Finalize();
    return 0;
}

Writing bcast.c


In [ ]:
!mpicc -o bcast bcast.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./bcast

[Processo 0] vetor recebido: 10 20 30 40 50 
[Processo 1] vetor recebido: 10 20 30 40 50 
[Processo 3] vetor recebido: 10 20 30 40 50 
[Processo 2] vetor recebido: 10 20 30 40 50 


### 2.2 `MPI_Scatter`

O processo raiz possui um **vetor** com `N` elementos (`N` = número de
processos × tamanho do bloco) e **divide** esse vetor em blocos contíguos e
de mesmo tamanho, enviando **um bloco diferente** para cada processo — o
processo de rank `i` recebe o `i`-ésimo bloco. É o padrão típico para
distribuir dados de entrada (linhas de uma matriz, elementos de um vetor a
processar) entre os processos de forma balanceada.

```c
int MPI_Scatter(const void* sendbuf, int sendcount, MPI_Datatype sendtype,
                 void* recvbuf, int recvcount, MPI_Datatype recvtype,
                 int root, MPI_Comm comm);
```
- `sendbuf`: vetor completo, significativo apenas na raiz.
- `sendcount`: quantidade de elementos **enviada a cada processo** (não o
  total do vetor).
- `recvbuf` / `recvcount`: buffer e quantidade de elementos que **cada
  processo**, incluindo a raiz, recebe.
- `root`: rank do processo que possui o vetor completo.

```python
comm.Scatter(vetor_completo, minha_parte, root=0)
```

> Quando os blocos não têm o mesmo tamanho para todos os processos, existe a
> variante `MPI_Scatterv`, que permite tamanhos e deslocamentos distintos.


In [ ]:
%%writefile scatter.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

BLOCO = 2  # cada processo recebe 2 elementos

if rank == 0:
    # vetor completo: [0, 1, 2, ..., size*BLOCO - 1]
    vetor_completo = np.arange(size * BLOCO, dtype='i')
else:
    vetor_completo = None

minha_parte = np.empty(BLOCO, dtype='i')
comm.Scatter(vetor_completo, minha_parte, root=0)

print(f"[Processo {rank}] recebi o bloco: {minha_parte}")

Writing scatter.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python scatter.py

[Processo 0] recebi o bloco: [0 1]
[Processo 1] recebi o bloco: [2 3]
[Processo 2] recebi o bloco: [4 5]
[Processo 3] recebi o bloco: [6 7]


Versão equivalente em **C**:

In [ ]:
%%writefile scatter.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

#define BLOCO 2

int main(int argc, char** argv) {
    int rank, size;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int *vetor_completo = NULL;
    if (rank == 0) {
        vetor_completo = (int*) malloc(size * BLOCO * sizeof(int));
        for (int i = 0; i < size * BLOCO; i++) vetor_completo[i] = i;
    }

    int minha_parte[BLOCO];
    MPI_Scatter(vetor_completo, BLOCO, MPI_INT,
                minha_parte, BLOCO, MPI_INT,
                0, MPI_COMM_WORLD);

    printf("[Processo %d] recebi o bloco: ", rank);
    for (int i = 0; i < BLOCO; i++) printf("%d ", minha_parte[i]);
    printf("\n");

    if (rank == 0) free(vetor_completo);

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -o scatter scatter.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./scatter

### 2.3 `MPI_Gather`

Operação inversa ao `Scatter`: cada processo envia um **vetor** local e a
raiz recebe todos, **concatenados na ordem dos ranks** (o bloco do processo 0
vem primeiro, depois o do processo 1, e assim por diante), formando um único
vetor maior. É o padrão típico para reunir resultados parciais calculados
localmente por cada processo em uma única estrutura, geralmente ao final de
um cálculo paralelo.

```c
int MPI_Gather(const void* sendbuf, int sendcount, MPI_Datatype sendtype,
                void* recvbuf, int recvcount, MPI_Datatype recvtype,
                int root, MPI_Comm comm);
```
- `sendbuf` / `sendcount`: vetor local e sua quantidade de elementos,
  enviados por **cada** processo.
- `recvbuf`: vetor onde a raiz armazena tudo o que recebeu — deve ter espaço
  para `recvcount × número de processos` elementos.
- `recvcount`: quantidade de elementos recebida **de cada** processo
  (significativo apenas na raiz).
- `root`: rank do processo que coleta os dados.

```python
comm.Gather(meu_vetor, vetor_coletado, root=0)
```

> Assim como no `Scatter`, existe a variante `MPI_Gatherv` para blocos de
> tamanhos diferentes entre os processos.


In [ ]:
%%writefile gather.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

BLOCO = 2
meu_vetor = np.array([rank, rank ** 2], dtype='i')  # vetor local

if rank == 0:
    vetor_coletado = np.empty(size * BLOCO, dtype='i')
else:
    vetor_coletado = None

comm.Gather(meu_vetor, vetor_coletado, root=0)

if rank == 0:
    print(f"[Processo 0] vetor coletado: {vetor_coletado}")

Writing gather.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python gather.py

[Processo 0] vetor coletado: [0 0 1 1 2 4 3 9]


Versão equivalente em **C**:

In [ ]:
%%writefile gather.c
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>

#define BLOCO 2

int main(int argc, char** argv) {
    int rank, size;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int meu_vetor[BLOCO] = {rank, rank * rank};

    int *vetor_coletado = NULL;
    if (rank == 0) {
        vetor_coletado = (int*) malloc(size * BLOCO * sizeof(int));
    }

    MPI_Gather(meu_vetor, BLOCO, MPI_INT,
               vetor_coletado, BLOCO, MPI_INT,
               0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("[Processo 0] vetor coletado: ");
        for (int i = 0; i < size * BLOCO; i++) printf("%d ", vetor_coletado[i]);
        printf("\n");
        free(vetor_coletado);
    }

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -o gather gather.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./gather

### 2.4 `MPI_Reduce`

Combina, **elemento a elemento**, os vetores de todos os processos usando uma
operação associativa (soma, produto, máximo, mínimo, ...) e entrega **apenas
à raiz** o vetor resultante — os demais processos não recebem o resultado.
É a forma padrão de agregar resultados parciais de um cálculo paralelo em um
único valor final (ex.: soma total, maior valor encontrado por qualquer
processo).

```c
int MPI_Reduce(const void* sendbuf, void* recvbuf, int count,
                MPI_Datatype datatype, MPI_Op op, int root, MPI_Comm comm);
```
- `sendbuf`: vetor local de cada processo, com o valor a ser combinado.
- `recvbuf`: vetor onde a raiz recebe o resultado combinado (ignorado nos
  demais processos).
- `count`: número de elementos combinados (a operação é aplicada posição a
  posição, ex.: `resultado[i] = op(vetor_p0[i], vetor_p1[i], ...)`).
- `op`: operação de redução a aplicar.
- `root`: rank do processo que recebe o resultado.

```python
comm.Reduce(meu_vetor, vetor_resultado, op=MPI.SUM, root=0)
```

Operações mais comuns: `MPI.SUM` (soma), `MPI.PROD` (produto), `MPI.MAX`
(máximo), `MPI.MIN` (mínimo) — também existem `MPI.LAND`/`MPI.LOR` (E/OU
lógico) e outras.


In [ ]:
%%writefile reduce.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

N = 3
meu_vetor = np.array([rank + 1, rank + 2, rank + 3], dtype='i')

if rank == 0:
    vetor_resultado = np.empty(N, dtype='i')
else:
    vetor_resultado = None

comm.Reduce(meu_vetor, vetor_resultado, op=MPI.SUM, root=0)

if rank == 0:
    print(f"[Processo 0] soma elemento a elemento: {vetor_resultado}")

Writing reduce.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python reduce.py

[Processo 0] soma elemento a elemento: [10 14 18]


Versão equivalente em **C**:

In [ ]:
%%writefile reduce.c
#include <mpi.h>
#include <stdio.h>

#define N 3

int main(int argc, char** argv) {
    int rank;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    int meu_vetor[N] = {rank + 1, rank + 2, rank + 3};
    int vetor_resultado[N];

    MPI_Reduce(meu_vetor, vetor_resultado, N, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        printf("[Processo 0] soma elemento a elemento: ");
        for (int i = 0; i < N; i++) printf("%d ", vetor_resultado[i]);
        printf("\n");
    }

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -o reduce reduce.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./reduce

### 2.5 `MPI_Allreduce`

Igual ao `MPI_Reduce`, mas o **vetor** combinado é entregue **a todos os
processos**, não apenas à raiz — ao final, cada processo tem uma cópia
idêntica do resultado. Conceitualmente equivale a um `Reduce` seguido de um
`Bcast`, porém a biblioteca MPI implementa essa sequência de forma otimizada
(um único algoritmo coletivo), evitando o custo de duas chamadas separadas.
É útil quando **todos** os processos precisam conhecer um valor global
(soma, máximo, etc.) para continuar seus próprios cálculos.

```c
int MPI_Allreduce(const void* sendbuf, void* recvbuf, int count,
                   MPI_Datatype datatype, MPI_Op op, MPI_Comm comm);
```
- Mesmos parâmetros do `MPI_Reduce`, exceto que **não há** parâmetro `root`
  (todos recebem o resultado, não apenas um processo específico).

```python
comm.Allreduce(meu_vetor, vetor_resultado, op=MPI.SUM)
```


In [ ]:
%%writefile allreduce.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

N = 3
meu_vetor = np.array([rank + 1, rank + 2, rank + 3], dtype='i')
vetor_resultado = np.empty(N, dtype='i')

comm.Allreduce(meu_vetor, vetor_resultado, op=MPI.SUM)

# Repare: TODOS os processos têm o vetor resultado, não só o rank 0
print(f"[Processo {rank}] soma total (todos sabem): {vetor_resultado}")

Writing allreduce.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python allreduce.py

[Processo 3] soma total (todos sabem): [10 14 18]
[Processo 1] soma total (todos sabem): [10 14 18]
[Processo 2] soma total (todos sabem): [10 14 18]
[Processo 0] soma total (todos sabem): [10 14 18]


Versão equivalente em **C**:

In [ ]:
%%writefile allreduce.c
#include <mpi.h>
#include <stdio.h>

#define N 3

int main(int argc, char** argv) {
    int rank;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    int meu_vetor[N] = {rank + 1, rank + 2, rank + 3};
    int vetor_resultado[N];

    MPI_Allreduce(meu_vetor, vetor_resultado, N, MPI_INT, MPI_SUM, MPI_COMM_WORLD);

    printf("[Processo %d] soma total (todos sabem): ", rank);
    for (int i = 0; i < N; i++) printf("%d ", vetor_resultado[i]);
    printf("\n");

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -o allreduce allreduce.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./allreduce

### 2.6 `MPI_Allgather`

Igual ao `MPI_Gather`, mas o **vetor** final coletado (concatenado na ordem
dos ranks) é entregue **a todos os processos**, não só à raiz — equivalente
a um `Gather` seguido de um `Bcast`, novamente otimizado internamente pelo
MPI. É muito usado em algoritmos distribuídos nos quais cada processo
precisa ter uma visão **completa e atualizada** dos dados parciais de todos
os outros para prosseguir com seu próprio cálculo (ex.: métodos iterativos,
simulações onde cada processo depende dos vizinhos).

```c
int MPI_Allgather(const void* sendbuf, int sendcount, MPI_Datatype sendtype,
                   void* recvbuf, int recvcount, MPI_Datatype recvtype,
                   MPI_Comm comm);
```
- Mesmos parâmetros do `MPI_Gather`, exceto que **não há** parâmetro `root`
  (o vetor concatenado vai para todos os processos).

```python
comm.Allgather(meu_vetor, vetor_coletado)
```


In [ ]:
%%writefile allgather.py
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

BLOCO = 2
meu_vetor = np.array([rank, rank ** 2], dtype='i')
vetor_coletado = np.empty(size * BLOCO, dtype='i')

comm.Allgather(meu_vetor, vetor_coletado)

print(f"[Processo {rank}] vejo todos os valores: {vetor_coletado}")

Writing allgather.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 4 python allgather.py

[Processo 0] vejo todos os valores: [0 0 1 1 2 4 3 9]
[Processo 1] vejo todos os valores: [0 0 1 1 2 4 3 9]
[Processo 3] vejo todos os valores: [0 0 1 1 2 4 3 9]
[Processo 2] vejo todos os valores: [0 0 1 1 2 4 3 9]


Versão equivalente em **C**:

In [ ]:
%%writefile allgather.c
#include <mpi.h>
#include <stdio.h>

#define BLOCO 2

int main(int argc, char** argv) {
    int rank, size;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int meu_vetor[BLOCO] = {rank, rank * rank};
    int vetor_coletado[BLOCO * 32];  // buffer com folga para até 32 processos

    MPI_Allgather(meu_vetor, BLOCO, MPI_INT,
                  vetor_coletado, BLOCO, MPI_INT,
                  MPI_COMM_WORLD);

    printf("[Processo %d] vejo todos os valores: ", rank);
    for (int i = 0; i < size * BLOCO; i++) printf("%d ", vetor_coletado[i]);
    printf("\n");

    MPI_Finalize();
    return 0;
}

In [ ]:
!mpicc -o allgather allgather.c
!mpirun --allow-run-as-root --oversubscribe -np 4 ./allgather

## 3. Resumo das funções apresentadas

| Função | Categoria | Efeito |
|---|---|---|
| `MPI_Barrier` | Sincronização | Bloqueia todos os processos até que todos cheguem à barreira |
| `MPI_Bcast` | Coletiva — distribuição | Raiz envia o mesmo dado a todos |
| `MPI_Scatter` | Coletiva — distribuição | Raiz distribui partes diferentes de um vetor |
| `MPI_Gather` | Coletiva — coleta | Raiz coleta um valor de cada processo, em ordem |
| `MPI_Reduce` | Coletiva — redução | Combina valores de todos os processos; resultado só na raiz |
| `MPI_Allreduce` | Coletiva — redução | Como o `Reduce`, mas resultado entregue a todos |
| `MPI_Allgather` | Coletiva — coleta | Como o `Gather`, mas resultado entregue a todos |


## 4. Exercícios

Implemente os exercícios abaixo em **mpi4py** (ou em C, se preferir), usando o
mesmo padrão `%%writefile` + `mpirun` utilizado nos exemplos acima. Rode com
pelo menos **4 processos**.

### Exercício 1 — Barreira com medição de tempo

Cada processo deve "trabalhar" por um tempo aleatório entre 1 e 5 segundos
(`time.sleep`). Utilize `MPI_Barrier` para garantir que **todos os processos
terminem essa etapa** antes de iniciar uma segunda fase, na qual cada processo
imprime `"Fase 2 iniciada pelo processo X"`. Meça e imprima, no processo 0, o
tempo total decorrido entre o início do programa e a passagem pela barreira.

### Exercício 2 — Distribuição e redução de um vetor

O processo raiz (rank 0) deve criar um vetor com `N` números inteiros
(`N` = número de processos, com valores, por exemplo, de 1 a N). Use
`MPI_Scatter` para distribuir **um número para cada processo**. Cada processo
deve elevar o número recebido ao quadrado. Em seguida, utilize `MPI_Gather`
para reunir todos os resultados na raiz e `MPI_Reduce` para calcular, também
na raiz, a **soma total** dos quadrados. Imprima o vetor original, o vetor de
quadrados coletado e a soma total.

### Exercício 3 — Estatísticas distribuídas com Allreduce

Cada processo deve gerar um valor aleatório (por exemplo, um número inteiro
entre 0 e 100 — use uma semente diferente por processo, baseada no `rank`).
Utilize `MPI_Allreduce` **três vezes** (ou combine as operações) para que
**todos os processos** fiquem sabendo, ao final:

- a **soma** de todos os valores gerados;
- o **valor máximo** entre todos os processos;
- o **valor mínimo** entre todos os processos.

Cada processo deve imprimir seu próprio valor gerado seguido da soma, máximo e
mínimo calculados — todos os processos devem imprimir os mesmos três
resultados finais, confirmando que a informação chegou a todos.

### Exercício 4 — Soma de matrizes distribuída

Sejam **A** e **B** duas matrizes quadradas de ordem `N` (`N` = número de
processos), preenchidas pelo processo raiz com valores de sua escolha
(por exemplo, `A[i][j] = i + j` e `B[i][j] = i - j`). O objetivo é calcular
`C = A + B` de forma distribuída, seguindo os passos:

1. Na raiz, armazene `A` e `B` como vetores unidimensionais (cada matriz com
   `N × N` elementos, com as linhas concatenadas — *row-major*).
2. Use `MPI_Scatter` **duas vezes** para distribuir **uma linha de A e a
   linha correspondente de B** para cada processo (cada processo deve
   receber `N` elementos de `A` e `N` elementos de `B`).
3. Cada processo calcula localmente a soma elemento a elemento de sua linha:
   `linha_C[j] = linha_A[j] + linha_B[j]`.
4. Use `MPI_Gather` para reunir, na raiz, todas as linhas calculadas,
   remontando a matriz `C` completa.
5. No processo raiz, imprima as matrizes `A`, `B` e `C` de forma legível
   (uma linha da matriz por linha impressa).

**Desafio adicional:** modifique o programa para que, além de calcular `C`,
cada processo também contribua com a soma de todos os elementos de sua linha
de `C` para um `MPI_Reduce` com `MPI.SUM`, de modo que a raiz imprima também
a **soma total de todos os elementos da matriz resultante**.
